# 📡 Real-Time Live Testing Interface — Dhaka 24h-Ahead PM2.5 Forecaster

This interactive research notebook fetches **live, real-time air quality and meteorological measurements for Dhaka, Bangladesh** from Open-Meteo's open APIs and executes the **Hybrid Ridge-Residual Champion Model** ($R^2 = 0.8650$, $\text{RMSE} = 21.66\ \mu\text{g/m}^3$).

### Capabilities for Journal Paper Publication:
1. **Zero-Configuration API Stream:** Automatically fetches hourly $PM_{2.5}$, temperature, humidity, wind speed, and rainfall for Dhaka (`23.8103° N, 90.4125° E`).
2. **Real-Time Automated Feature Engineering:** Computes the Core 92 Causal Feature Set (`pm25_curr`, causal lags, EMAs, rolling statistics, 24h growth rates, and 24h meteorological averages).
3. **Live 24h-Ahead Daily Average Forecast:** Outputs the predicted 24-hour ahead daily average concentration and classifies it into **WHO/EPA Air Quality Index (AQI) Severity Bands**.
4. **Publication-Ready Visualization:** Generates and saves a high-resolution real-time forecast figure (`dhaka_live_forecast.png`).


In [ ]:
# 1. Environment Setup & Model Loading / Auto-Training
import os, json, pickle, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
SEED = 42

# Attempt to load saved champion model artifact, or auto-fit if version mismatch / running standalone
model_paths = [
    'models/live_hybrid_champion.pkl',
    'live_hybrid_champion.pkl',
    '/home/user/models/live_hybrid_champion.pkl',
    '/home/user/Dhaka_PM25_Live_Verification_Suite/models/live_hybrid_champion.pkl'
]
model_path = next((p for p in model_paths if os.path.exists(p)), None)
artifact = None
if model_path:
    try:
        with open(model_path, 'rb') as f:
            artifact = pickle.load(f)
        print("✓ Loaded saved Hybrid Ridge-Residual Champion Model artifact successfully!")
    except Exception as e:
        print(f"Note: Saved pickle was created with a different Scikit-Learn version ({e}). Auto-fitting for your local Python environment...")

if not artifact:
    print("Training Hybrid Ridge-Residual Champion Model on historical data (~3 seconds)...")
    data_paths = [
        'data/final_dataset_clean.csv',
        'final_dataset_clean.csv',
        '/home/user/uploads/final_dataset_clean.csv',
        '/kaggle/input/datasets/begumluthfunnesa/thesis/final_dataset_clean.csv'
    ]
    data_path = next((p for p in data_paths if os.path.exists(p)), 'data/final_dataset_clean.csv')
    df_clean = pd.read_csv(data_path)
    df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])
    df_clean = df_clean.sort_values('datetime').reset_index(drop=True)
    
    df = df_clean.copy()
    df = df[df['pm25'] >= 1].reset_index(drop=True)
    p995 = df['pm25'].quantile(0.995)
    df.loc[df['pm25'] > p995, 'pm25'] = p995
    
    pm25 = df['pm25']
    df['pm25_curr'] = pm25
    
    for lag in [1, 2, 3, 4, 5, 6, 8, 12, 16, 20, 24, 36, 48, 72, 120, 168]:
        df[f'pm25_lag_{lag}'] = pm25.shift(lag)
    for span in [3, 6, 12, 24, 48, 72, 168]:
        df[f'pm25_ema_{span}'] = pm25.ewm(span=span, adjust=False).mean()
    for w in [3, 6, 12, 24, 48, 72, 168]:
        base = pm25.rolling(w, min_periods=max(1, int(w*0.5)))
        df[f'pm25_roll_mean_{w}']   = base.mean()
        df[f'pm25_roll_std_{w}']    = base.std()
        df[f'pm25_roll_max_{w}']    = base.max()
        df[f'pm25_roll_min_{w}']    = base.min()
    
    df['target'] = pm25.shift(-24).rolling(24, min_periods=24).mean()
    df['roll24_diff_24'] = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(24)
    df['roll24_diff_48'] = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(48)
    df['roll24_diff_1']  = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(1)
    df['roll24_accel']   = df['roll24_diff_24'] - df['roll24_diff_24'].shift(24)
    df['roll24_growth']  = df['pm25_roll_mean_24'] / (df['pm25_roll_mean_24'].shift(24) + 1e-5)
    
    df['hour']  = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['doy']   = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24.0)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24.0)
    df['doy_sin']  = np.sin(2*np.pi*df['doy']/365.25)
    df['doy_cos']  = np.cos(2*np.pi*df['doy']/365.25)
    
    for col in ['temperature', 'humidity', 'wind_speed', 'rainfall']:
        for lag in [1, 6, 12, 24, 48]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll24'] = df[col].rolling(24, min_periods=12).mean()
    
    DROP_ALWAYS = ['datetime', 'target']
    FEATURE_COLS = [c for c in df.columns if c not in DROP_ALWAYS]
    df_sel = df[FEATURE_COLS + ['target']].dropna().reset_index(drop=True)
    
    X_all = df_sel[FEATURE_COLS].values; y_all = df_sel['target'].values
    scaler = StandardScaler()
    X_all_s = scaler.fit_transform(X_all)
    
    m_ridge = RidgeCV(alphas=np.logspace(-2, 6, 50))
    m_ridge.fit(X_all_s, y_all)
    res_all = y_all - m_ridge.predict(X_all_s)
    
    m_res = HistGradientBoostingRegressor(loss='squared_error', max_iter=400, learning_rate=0.03, max_depth=4, min_samples_leaf=15, random_state=SEED)
    m_res.fit(X_all, res_all)
    
    artifact = {
        'scaler': scaler,
        'ridge_model': m_ridge,
        'residual_tree_model': m_res,
        'feature_cols': FEATURE_COLS,
        'p995': p995
    }
    print("✓ Model trained and ready for live real-time prediction!")


### 2. Fetch Real-Time Live PM2.5 & Meteorology for Dhaka (Open-Meteo API)
We pull the last 14 days of hourly measurements up to the current UTC timestamp for Dhaka (`lat=23.8103, lon=90.4125`).


In [ ]:
def fetch_live_openmeteo_dhaka():
    url_aq = 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=23.8103&longitude=90.4125&hourly=pm2_5&past_days=14&forecast_days=1'
    url_wx = 'https://api.open-meteo.com/v1/forecast?latitude=23.8103&longitude=90.4125&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m,rain&past_days=14&forecast_days=1'
    
    req_aq = urllib.request.urlopen(url_aq)
    data_aq = json.loads(req_aq.read().decode('utf-8'))
    
    req_wx = urllib.request.urlopen(url_wx)
    data_wx = json.loads(req_wx.read().decode('utf-8'))
    
    times = pd.to_datetime(data_aq['hourly']['time'])
    pm_vals = data_aq['hourly']['pm2_5']
    
    df_live = pd.DataFrame({
        'datetime': times,
        'pm25': pm_vals,
        'temperature': data_wx['hourly']['temperature_2m'],
        'humidity': data_wx['hourly']['relative_humidity_2m'],
        'wind_speed': data_wx['hourly']['wind_speed_10m'],
        'rainfall': data_wx['hourly']['rain']
    })
    df_live = df_live.dropna().sort_values('datetime').reset_index(drop=True)
    return df_live

df_live = fetch_live_openmeteo_dhaka()
print(f"✓ Loaded {len(df_live)} real-time hourly records up to {df_live['datetime'].iloc[-1]} UTC.")
print("Latest 5 Hourly Readings for Dhaka:")
print(df_live.tail(5).to_string(index=False))


### 3. Generate Live Causal Features & Predict 24-Hour Ahead Concentration


In [ ]:
def engineer_live_features(df_input, feature_cols):
    df = df_input.copy()
    pm25 = df['pm25']
    df['pm25_curr'] = pm25
    
    for lag in [1, 2, 3, 4, 5, 6, 8, 12, 16, 20, 24, 36, 48, 72, 120, 168]:
        df[f'pm25_lag_{lag}'] = pm25.shift(lag)
        
    for span in [3, 6, 12, 24, 48, 72, 168]:
        df[f'pm25_ema_{span}'] = pm25.ewm(span=span, adjust=False).mean()
        
    for w in [3, 6, 12, 24, 48, 72, 168]:
        base = pm25.rolling(w, min_periods=max(1, int(w*0.5)))
        df[f'pm25_roll_mean_{w}']   = base.mean()
        df[f'pm25_roll_std_{w}']    = base.std()
        df[f'pm25_roll_max_{w}']    = base.max()
        df[f'pm25_roll_min_{w}']    = base.min()
        
    df['roll24_diff_24'] = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(24)
    df['roll24_diff_48'] = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(48)
    df['roll24_diff_1']  = df['pm25_roll_mean_24'] - df['pm25_roll_mean_24'].shift(1)
    df['roll24_accel']   = df['roll24_diff_24'] - df['roll24_diff_24'].shift(24)
    df['roll24_growth']  = df['pm25_roll_mean_24'] / (df['pm25_roll_mean_24'].shift(24) + 1e-5)
    
    df['hour']  = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['doy']   = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24.0)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24.0)
    df['doy_sin']  = np.sin(2*np.pi*df['doy']/365.25)
    df['doy_cos']  = np.cos(2*np.pi*df['doy']/365.25)
    
    for col in ['temperature', 'humidity', 'wind_speed', 'rainfall']:
        for lag in [1, 6, 12, 24, 48]:
            df[f'{col}_lag_{lag}'] = df[col].shift(lag)
        df[f'{col}_roll24'] = df[col].rolling(24, min_periods=12).mean()
        
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0
            
    df = df.bfill().ffill().fillna(0.0)
    return df

df_feat = engineer_live_features(df_live, artifact['feature_cols'])
latest_row = df_feat.iloc[-1:]
X_live   = latest_row[artifact['feature_cols']].values
X_live_s = artifact['scaler'].transform(X_live)

pred_ridge = artifact['ridge_model'].predict(X_live_s)[0]
pred_res   = artifact['residual_tree_model'].predict(X_live)[0]
pred_24h   = max(5.0, pred_ridge + pred_res)

print("="*70)
print("=== LIVE 24-HOUR AHEAD DAILY AVERAGE PM2.5 FORECAST FOR DHAKA ===")
print("="*70)
print(f"Timestamp (UTC)                 : {df_live['datetime'].iloc[-1]}")
print(f"Current Hourly PM2.5            : {df_live['pm25'].iloc[-1]:.1f} µg/m³")
print(f"Current Temperature / Hum / Wind: {df_live['temperature'].iloc[-1]:.1f}°C | {df_live['humidity'].iloc[-1]:.0f}% | {df_live['wind_speed'].iloc[-1]:.1f} km/h")
print(f"Stage 1 Linear Autoregression   : {pred_ridge:.1f} µg/m³")
print(f"Stage 2 Non-Linear Correction   : {pred_res:+.1f} µg/m³")
print("-" * 70)
print(f"FORECASTED 24H DAILY AVERAGE    : {pred_24h:.1f} µg/m³")
print("="*70)



### 4. Public Health Severity Classification & Publication-Ready Chart


In [ ]:
def classify_aqi_band(pm25_val):
    if pm25_val <= 50:
        return 'Good (0-50)', '#2CA02C', 'Air quality is satisfactory; air pollution poses little or no risk.'
    elif pm25_val <= 100:
        return 'Moderate (51-100)', '#FFC107', 'Air quality is acceptable; sensitive individuals should monitor prolonged exposure.'
    elif pm25_val <= 150:
        return 'Unhealthy for Sensitive Groups (101-150)', '#FF9800', 'Sensitive groups may experience health effects. General public is less likely to be affected.'
    elif pm25_val <= 200:
        return 'Unhealthy (151-200)', '#F44336', 'Everyone may begin to experience health effects; sensitive groups may experience serious effects.'
    else:
        return 'Very Unhealthy / Hazardous (200+)', '#8E24AA', 'Health warnings of emergency conditions. Entire population is likely to be affected.'

band_label, band_color, band_desc = classify_aqi_band(pred_24h)
print(f"WHO / EPA AQI Band : {band_label}")
print(f"Health Advisory    : {band_desc}")

# Plot publication-ready chart
fig, ax = plt.subplots(figsize=(13, 5))
recent = df_live.iloc[-168:]
ax.plot(recent['datetime'], recent['pm25'], color='#185FA5', lw=2, label='Actual Live Hourly PM2.5 (Open-Meteo API)')

future_time = recent['datetime'].iloc[-1] + pd.Timedelta(hours=24)
ax.plot([recent['datetime'].iloc[-1], future_time],
        [recent['pm25'].iloc[-1], pred_24h],
        color='#D85A30', linestyle='--', lw=2.5, marker='o', label=f'24h-Ahead Forecast ({pred_24h:.1f} µg/m³)')

ax.axhline(50, color='#2CA02C', linestyle=':', alpha=0.7, label='Good Limit (50)')
ax.axhline(100, color='#FFC107', linestyle=':', alpha=0.7, label='Moderate Limit (100)')
ax.axhline(150, color='#FF9800', linestyle=':', alpha=0.7, label='Unhealthy for Sensitive Groups (150)')
ax.axhline(200, color='#F44336', linestyle=':', alpha=0.7, label='Unhealthy Limit (200)')

ax.set_title(f'Dhaka Real-Time PM2.5 Measurements & 24-Hour Ahead Forecast — [{band_label}]', fontweight='bold', fontsize=12)
ax.set_xlabel('UTC Date & Time', fontsize=11)
ax.set_ylabel('PM2.5 Concentration (µg/m³)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('/home/user/dhaka_live_forecast.png', dpi=200)
plt.show()
print("✓ Saved high-resolution figure: /home/user/dhaka_live_forecast.png")
